<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/4.2%20Building%20RAGs%20and%20Multi-Step%20Chains/3%20Tutorial%20-%20Adding%20External%20Data%20Processor%20to%20the%20Q%26A%20Application%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### OpenRouter API Key

This notebook uses OpenRouter through LangChain's OpenAI-compatible interface. Enter your OpenRouter API key when prompted. You do not need a separate OpenAI API key.


In [1]:
# Install pinned dependencies (Colab-ready; safe to re-run).
# Based on latest compatible versions
!pip install -q langchain-core==1.6.3 langchain-openai==1.6.2 langchain-pinecone==0.2.13 langchain-text-splitters==1.1.2 pinecone==7.3.0 python-dotenv==1.2.3 tiktoken==0.14.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.2 MB/s eta 0:00:00


## Tutorial: Building a Full RAG Q&A with Pinecone
We’ll ingest chunks into Pinecone, retrieve top‑k for a query, and answer with a grounded prompt.


In [2]:
import os
import time
from typing import List
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter OPENROUTER_API_KEY (hidden): ")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or getpass("Enter PINECONE_API_KEY (hidden): ")

# PineconeVectorStore reads the key from the environment rather than a variable.
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openai/gpt-4o-mini"
EMBEDDING_MODEL = "openai/text-embedding-3-small"

# Pinecone setup. Use a dedicated index and namespace so this tutorial does not mix data with others.
INDEX_NAME = os.getenv("PINECONE_INDEX", "lc-external-processor-demo")
NAMESPACE = "external-processor-demo"

from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=PINECONE_API_KEY)
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=os.getenv("PINECONE_REGION", "us-east-1"))
    )

while not pc.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(1)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model=MODEL, api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL, temperature=0, seed=42)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL, dimensions=1536)
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings, namespace=NAMESPACE)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)


Enter OPENROUTER_API_KEY (hidden): ··········
Enter PINECONE_API_KEY (hidden): ··········


### Step 1: Ingest documents into Pinecone
We’ll split raw text into chunks and upsert them with metadata.


In [3]:
raw_corpus = (
    "LangChain is a framework for building LLM applications with prompts, chains, tools, and agents. "
    "It integrates with vector databases like Pinecone to enable retrieval‑augmented generation. "
    "Prompt templates, chunking, and retrievers are key to reliable Q&A."
)
chunks = text_splitter.split_text(raw_corpus)
metas = [{"source": "local", "chunk": i} for i in range(len(chunks))]
vectorstore.add_texts(texts=chunks, metadatas=metas)
print(f"Upserted {len(chunks)} chunks to index '{INDEX_NAME}'.")


Upserted 1 chunks to index 'lc-external-processor-demo'.


### Step 2: Build a grounded Q&A prompt
We’ll keep it extractive and cite the `source` in metadata when helpful.


In [5]:
qa_template = (
    "You are a precise assistant. Use the CONTEXT to answer the QUESTION.\n"
    "If not answerable from the CONTEXT, say: I don't know.\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n"
    "ANSWER:"
)
qa_prompt = PromptTemplate.from_template(qa_template)
qa_chain = qa_prompt | llm

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def query_rag(question: str) -> str:
    docs = retriever.invoke(question)
    if not docs:
        return "I don't know."
    context = "\n\n".join(d.page_content for d in docs)
    response = qa_chain.invoke({"context": context, "question": question})
    return response.content if isinstance(response.content, str) else str(response.content)

print(query_rag("Name two LangChain building blocks."))


Prompt templates and retrievers.


### Step 3: Ask a few questions
We’ll test multiple questions and observe grounded behavior.


In [6]:
for q in [
    "Summarize what LangChain is in 1 line.",
    "What enables RAG with LangChain and Pinecone?",
    "What dataset does LangChain use to train?"
]:
    print("Q:", q)
    print(query_rag(q))
    print("-")


Q: Summarize what LangChain is in 1 line.
LangChain is a framework for building LLM applications that integrates prompts, chains, tools, and agents, along with vector databases for enhanced retrieval and generation.
-
Q: What enables RAG with LangChain and Pinecone?
Retrieval-augmented generation (RAG) with LangChain and Pinecone is enabled by the integration of vector databases, along with the use of prompt templates, chunking, and retrievers.
-
Q: What dataset does LangChain use to train?
I don't know.
-
